In [ ]:
### Homework: Agentic RAG

In [ ]:
# Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
#Q1. How many lesson pages

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
len(files)

72

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
documents[0]
len(documents)

72

In [7]:
# Q2. Indexing and searching

In [8]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [9]:
type(index.docs)

list

In [10]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={'content': 1.0},
    num_results=3
)

search_results
print(search_results[0]['filename'])


01-agentic-rag/lessons/14-agentic-loop.md


In [11]:
# Q3. RAG

In [12]:
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-20 20:24:01--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-20 20:24:01 (204 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [13]:
from rag_helper import RAGBase

In [14]:
class HomeworkRAG(RAGBase):
    def search(self, query):
        return self.index.search(
            query=query,
            boost_dict={'content': 1.0},
            num_results=3
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(f"Content: {doc['content']}")
            lines.append('')

        return '\n'.join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)

        response = self.llm(prompt)

        answer = response.choices[0].message.content
        usage = response.usage.prompt_tokens

        return answer, usage
        # return response

In [15]:
assistant = HomeworkRAG(
    index=index,
    llm_client=openai_client,
)

answer, usage = assistant.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

type(assistant)


print(answer)
print(usage)

The loop keeps calling the model by checking whether the model returned any `function_call` items. After each call:

1. It sends the current `messages` history to the model.
2. If the response contains a function call, it runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
3. At the end of the iteration, if `has_function_calls == False`, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.
5731


In [16]:
# Q4. Chunking 

In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [18]:
type(chunks)
len(chunks)

295

In [19]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [20]:
# Q5. RAG with chunking

In [21]:
index_ch = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_ch.fit(chunks)

In [22]:
len(index_ch.docs)

295

In [23]:
assistant_chunks = HomeworkRAG(
    index=index_ch,
    llm_client=openai_client,
)
answer_chunks, usage_chunks = assistant_chunks.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

In [28]:
assistant_chunks.index

In [25]:
print(answer)
print(usage)

The loop keeps calling the model by checking whether the model returned any `function_call` items. After each call:

1. It sends the current `messages` history to the model.
2. If the response contains a function call, it runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
3. At the end of the iteration, if `has_function_calls == False`, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.
5731


In [29]:
# Q6. Turning it into an agent


In [30]:
! uv add toyaikit

Resolved 128 packages in 1.19s                                       
Installed 7 packages in 35ms                                     
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [31]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [55]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index_ch.search(
        query,
        num_results=5,
        boost_dict={'content': 1.0},
    )

In [56]:
print(search)

<function search at 0x123106400>


In [60]:
ans = index_ch.search("olama")
len(ans)
ans[0]
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [61]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [63]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
